In [ ]:
import pandas as pd

df = pd.read_excel('../data/ttc-bus-delay-2024.xlsx')

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst 5 rows:")
df.head()

In [ ]:
# Cell 2 - Load data cleanly with all libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_excel('../data/ttc-bus-delay-2024.xlsx')

print("Shape:", df.shape)
print("\nDataset loaded successfully!")

In [ ]:
# Cell 3 - Basic information about the dataset
print("=== DATASET INFO ===")
print(df.info())

In [ ]:
# Cell 4 - Check missing values
print("=== MISSING VALUES ===")
missing = df.isnull().sum()
missing_percent = (df.isnull().sum() / len(df)) * 100
missing_table = pd.DataFrame({
    'Missing Count': missing,
    'Missing Percent': missing_percent
})
print(missing_table[missing_table['Missing Count'] > 0])

In [ ]:
# Cell 5 - Drop rows with invalid delay values
# Negative delay minutes make no sense - remove them
df = df[df['Min Delay'] >= 0]

# Drop rows where Route is missing - it is a key feature
df = df.dropna(subset=['Route'])

print("Rows with invalid or missing values dropped.")
print("New shape:", df.shape)

In [ ]:
# Cell 6 - Fill missing Direction with 'Unknown'
# We keep these rows because other features are still useful
df['Direction'] = df['Direction'].fillna('Unknown')

print("Direction missing values filled with 'Unknown'")
print("\nDirection value counts:")
print(df['Direction'].value_counts())

In [ ]:
# Cell 7 - Create target variable
# Standard transport rule: more than 5 minutes late = delayed
# This is the industry standard punctuality threshold

print("Min Delay distribution:")
print(df['Min Delay'].describe())

# 1 = Delayed (more than 5 minutes late)
# 0 = On Time (5 minutes or less)
df['delayed'] = (df['Min Delay'] > 5).astype(int)

print("\nNew target column 'delayed':")
print(df['delayed'].value_counts())
print("\nClass balance (%):")
print(df['delayed'].value_counts(normalize=True) * 100)

In [ ]:
# Cell 8 - Extract hour from Time column
# Time is in HH:MM format - we extract just the hour
df['hour'] = df['Time'].str.split(':').str[0].astype(int)

print("Hour column sample:")
print(df['hour'].value_counts().sort_index())

In [ ]:
# Cell 9 - Extract month from Date column
df['Date'] = pd.to_datetime(df['Date'])
df['month'] = df['Date'].dt.month
df['month_name'] = df['Date'].dt.month_name()

print("\nDay of week counts:")
print(df['Day'].value_counts())

print("\nMonth counts:")
print(df['month_name'].value_counts())

In [ ]:
# Cell 10 - Create peak hour and weekend flags
# Peak hours: morning 7-9am and evening 3-6pm
df['is_peak_hour'] = df['hour'].apply(
    lambda x: 1 if (7 <= x <= 9) or (15 <= x <= 18) else 0
)

# Weekend flag
df['is_weekend'] = df['Day'].apply(
    lambda x: 1 if x in ['Saturday', 'Sunday'] else 0
)

print("Peak hour distribution:")
print(df['is_peak_hour'].value_counts())

print("\nWeekend distribution:")
print(df['is_weekend'].value_counts())

In [ ]:
# Cell 11 - Top incident types
print("=== INCIDENT TYPES ===")
print(df['Incident'].value_counts())

In [ ]:
# Cell 12 - Top 10 most frequent routes
print("=== TOP 10 ROUTES BY FREQUENCY ===")
print(df['Route'].value_counts().head(10))

In [ ]:
# Cell 13 - Select only the columns we need for modelling
# Drop columns that are IDs, raw timestamps, or redundant

features_to_keep = [
    'Route',          # Bus route number
    'hour',           # Hour of day - extracted from Time
    'Day',            # Day of week
    'Incident',       # Type of incident - like reason for delay
    'Direction',      # Direction of travel N/S/E/W
    'Min Gap',        # Gap to next bus in minutes
    'is_peak_hour',   # 1 if rush hour, 0 otherwise
    'is_weekend',     # 1 if weekend, 0 if weekday
    'month',          # Month of year
    'delayed'         # TARGET variable - 1=delayed, 0=on time
]

df_model = df[features_to_keep].copy()

print("Final dataset shape:", df_model.shape)
print("\nMissing values in final dataset:")
print(df_model.isnull().sum())
print("\nFirst 3 rows:")
df_model.head(3)

In [ ]:
# Plot 1 - Class Balance
plt.figure(figsize=(6, 4))
df_model['delayed'].value_counts().plot(
    kind='bar',
    color=['#2ecc71', '#e74c3c']
)
plt.title('Class Balance — On Time vs Delayed')
plt.xlabel('0 = On Time, 1 = Delayed')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('../notebooks/ttc_plot1_class_balance.png')
plt.show()
print("Plot 1 saved!")

In [ ]:
# Plot 2 - Top Incident Types causing delays
plt.figure(figsize=(10, 5))
df_model['Incident'].value_counts().head(8).plot(
    kind='barh',
    color='steelblue'
)
plt.title('Top 8 Incident Types')
plt.xlabel('Count')
plt.tight_layout()
plt.savefig('../notebooks/ttc_plot2_incident_types.png')
plt.show()
print("Plot 2 saved!")

In [ ]:
# Plot 3 - Delay Rate by Hour of Day
plt.figure(figsize=(12, 5))
df_model.groupby('hour')['delayed'].mean().plot(
    kind='line',
    marker='o',
    color='coral'
)
plt.title('Delay Rate by Hour of Day')
plt.xlabel('Hour (24hr format)')
plt.ylabel('Delay Rate (0 to 1)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../notebooks/ttc_plot3_delay_by_hour.png')
plt.show()
print("Plot 3 saved!")

In [ ]:
# Plot 4 - Delay Rate by Day of Week
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday',
             'Friday', 'Saturday', 'Sunday']

plt.figure(figsize=(10, 5))
day_delay = df_model.groupby('Day')['delayed'].mean()
day_delay = day_delay.reindex(day_order)
day_delay.plot(kind='bar', color='teal')
plt.title('Delay Rate by Day of Week')
plt.xlabel('Day')
plt.ylabel('Delay Rate')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('../notebooks/ttc_plot4_delay_by_day.png')
plt.show()
print("Plot 4 saved!")

In [ ]:
# Plot 5 - Delay Rate by Incident Type
plt.figure(figsize=(10, 5))
incident_delay = df_model.groupby('Incident')['delayed'].mean().sort_values(ascending=False)
incident_delay.plot(kind='bar', color='mediumpurple')
plt.title('Delay Rate by Incident Type')
plt.xlabel('Incident Type')
plt.ylabel('Delay Rate')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('../notebooks/ttc_plot5_delay_by_incident.png')
plt.show()
print("Plot 5 saved!")

In [ ]:
# Plot 6 - Top 10 Most Delayed Routes
plt.figure(figsize=(10, 5))
route_delay = df_model.groupby('Route')['delayed'].mean()
top_routes = route_delay.sort_values(ascending=False).head(10)
top_routes.plot(kind='bar', color='salmon')
plt.title('Top 10 Most Delayed Bus Routes')
plt.xlabel('Route Number')
plt.ylabel('Delay Rate')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('../notebooks/ttc_plot6_top_routes.png')
plt.show()
print("Plot 6 saved!")

In [ ]:
# Cell 20 - Encode categorical columns
# Models need numbers - convert text to numbers using LabelEncoder
from sklearn.preprocessing import LabelEncoder

df_encoded = df_model.copy()

categorical_cols = ['Day', 'Incident', 'Direction']

le = LabelEncoder()
for col in categorical_cols:
    df_encoded[col] = le.fit_transform(df_encoded[col].astype(str))

print("Encoding done!")
print("\nFirst 3 rows after encoding:")
df_encoded.head(3)

In [ ]:
# Cell 21 - Save cleaned and encoded dataset
df_encoded.to_csv('../data/ttc_cleaned.csv', index=False)

print("Cleaned dataset saved to ../data/ttc_cleaned.csv")
print("Final shape:", df_encoded.shape)
print("\nReady for model building!")